In [ ]:
cuda_available = T.cuda.is_available()
mps_available = T.backends.mps.is_available()

print(f"CUDA (NVIDIA) available: {cuda_available}")
print(f"MPS (Apple Silicon) available: {mps_available}")
print("-" * 30)

# Determine the best available device
if cuda_available:
    device = T.device("cuda")
    print(f"Using device: {T.cuda.get_device_name(0)}")
elif mps_available:
    device = T.device("mps")
    print("Using device: Apple Silicon GPU (MPS)")
else:
    device = T.device("cpu")
    print("Running on CPU")

# print('Current NN architecture:', (state_shape[0], fc1_dims, fc2_dims, fc3_dims, len(Action)))

#### Best param save/load

In [ ]:
if IN_COLAB:
    base_dir = Path("/content/drive/MyDrive/Colab Notebooks")
else:
    # Path(".") points to your current working directory on both Mac and Windows
    base_dir = Path(".") 

# Path combining handles the slashes automatically (e.g., '\' for Windows, '/' for Mac/Colab)
best_param_loc = base_dir / "Optimized hyperparameters" / "ausgrid123-mali-SUPER.json"

# Save best Optuna parameters to file
with open(best_param_loc, "w") as f:
    json.dump(best_params, f, indent=4)

print(f"Best parameters saved to {best_param_loc}")
print(best_params)


In [ ]:
if IN_COLAB:
    best_param_loc = "/content/Household-RL/Optimized hyperparameters/N011Fc66Bu95Bk20PoskusZVecjimNN-AUSGRID.json"
else:
    best_param_loc = "Optimized hyperparameters/N001Fc100Bu95Bk20velikNN-ausgrid123.json"
# Load best parameters from file
with open(best_param_loc, "r") as f:
    best_params = json.load(f)

gamma = best_params['gamma']
lr = best_params['lr']
epsilon_start = best_params['epsilon_start']
epsilon_end = best_params['epsilon_end']
#epsilon_decay = best_params['epsilon_decay']
exploration_fraction = best_params['exploration_fraction']
total_steps = OPT_EPISODES * OPT_HORIZON
steps_to_decay = total_steps * exploration_fraction
batch_size = best_params['batch_size']
fc1_dims = 512 #best_params['fc1_dims']
fc2_dims = 256 #best_params['fc2_dims']
fc3_dims = 128 #best_params['fc3_dims']
replace_target = best_params['replace_target']
weight_decay = best_params['weight_decay']

print("Loaded best parameters from BestParam1.json")
print(best_params)


#### Optimization

In [ ]:
# Optimization horizons
OPT_EPISODES = 35
OPT_HORIZON = KorakovNaDan * 14


def objective(trial):
    gamma_t = trial.suggest_float("gamma", 0.8, 0.99999, log=True)
    lr_t = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    epsilon_start_t = trial.suggest_float("epsilon_start", 0.8, 1.0)
    epsilon_end_t = trial.suggest_float("epsilon_end", 0.01, 0.15)
    total_steps = OPT_EPISODES * OPT_HORIZON

    # Optimize the *fraction* of the total steps where epsilon should hit its minimum.
    # For example, we want exploration to end somewhere between 30% and 80% into the trial.
    exploration_fraction = trial.suggest_float("exploration_fraction", 0.3, 0.8)
    steps_to_decay = total_steps * exploration_fraction
     # Calculate the precise linear decay rate per step
    epsilon_decay_t = (epsilon_start_t - epsilon_end_t) / steps_to_decay
    #epsilon_decay_t = trial.suggest_float("epsilon_decay", 1e-6, 2e-4, log=True)
    batch_size_t = trial.suggest_categorical("batch_size", [32, 64, 96, 128])
    fc1_t = 512 * 1        #trial.suggest_categorical("fc1_dims", [128, 256, 512])
    fc2_t = 256 * 1       #trial.suggest_categorical("fc2_dims", [64, 128, 256])
    fc3_t = 128 * 1     #trial.suggest_categorical("fc3_dims", [32, 64, 128])
    replace_target_t = trial.suggest_int("replace_target", KorakovNaDan * 3, KorakovNaDan * 30)
    weight_decay_t = trial.suggest_float("weight_decay", 0.0, 1e-3)

    env = build_dqn_env(
        dataset=train_data,
        dataset_norm=train_data_norm,
        episode_length=OPT_HORIZON,
        reset_mode="random",
        observation_mode="sliding_window",
    )

    state_shape = [int(env.observation_space.shape[0])]
    network_dims = [state_shape[0], fc1_t, fc2_t, fc3_t, len(Action)]

    agent = AgentDQN(
        gamma=gamma_t,
        epsilon=epsilon_start_t,
        lr=lr_t,
        state_shape=state_shape,
        network_dims=network_dims,
        batch_size=batch_size_t,
        eps_end=epsilon_end_t,
        eps_dec=epsilon_decay_t,
        weight_decay=weight_decay_t,
        replace_target=replace_target_t,
    )

    total_reward = 0.0

    for episode in range(OPT_EPISODES):
        obs, _ = env.reset(options={"reset_mode": "random"})
        episode_reward = 0.0

        for step in range(OPT_HORIZON):
            a = agent.choose_action(obs)
            next_obs, r, terminated, truncated, _ = env.step(a)
            done = bool(terminated or truncated)

            agent.store_transition(obs, a, r, next_obs, done)
            agent.learn()

            obs = next_obs
            episode_reward += r

            if done:
                break

    # Keep track of your episode rewards in a list
    episode_rewards = []

    for episode in range(OPT_EPISODES):
        # ... your environment loop ...
        episode_rewards.append(episode_reward)
        
        # Report rolling average for pruning
        rolling_score = sum(episode_rewards) / len(episode_rewards)
        trial.report(rolling_score, episode)
        
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Return only the performance of the mature agent (e.g., last 10 episodes)
    return sum(episode_rewards[-10:]) / 10


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=40),
)

print("Starting hyperparameter optimization...")
study.optimize(objective, n_trials=100, timeout=3600)

print("Optimization completed.")
print(f"Finished trials: {len(study.trials)}")
print(f"Best trial: {study.best_trial.number}")
print(f"Best value: {study.best_trial.value:.4f}")

best_params = study.best_trial.params

plot_optimization_history(study).show()
plot_param_importances(study).show()

In [ ]:
# apply best Optuna parameters
gamma = best_params['gamma']
lr = best_params['lr']
epsilon_start = best_params['epsilon_start']
epsilon_end = best_params['epsilon_end']
#epsilon_decay = best_params['epsilon_decay']
exploration_fraction = best_params['exploration_fraction']
total_steps = OPT_EPISODES * OPT_HORIZON
steps_to_decay = total_steps * exploration_fraction
epsilon_decay = (epsilon_start - epsilon_end) / steps_to_decay
batch_size = best_params['batch_size']
# fc1_dims = best_params['fc1_dims']
# fc2_dims = best_params['fc2_dims']
# fc3_dims = best_params['fc3_dims']
replace_target = best_params['replace_target']
weight_decay = best_params['weight_decay']

#### Testiranje sprememb med enakimi funkcijami (vpliv naključnosti)

In [ ]:
nagrada_seznam = []
placilo_seznam = []
baterija_seznam = []
NagradaKapaciteta_seznam = []
NagradaSprememba_seznam = []
NagradaPlacilo_seznam = []

train_agent, train_env = Learning_DQN(ponovitev = 1, reset = True)
nagradaDQN , placiloDQN, baterijaDQN, NagradaKapaciteta, NagradaSprememba, NagradaPlacilo = Learning_DQN(ucenje = False, eval_epsilon=0.05)

nagrada_top = copy.deepcopy(nagradaDQN)
placilo_top = copy.deepcopy(placiloDQN)
baterija_top = copy.deepcopy(baterijaDQN)
NagradaKapaciteta_top = copy.deepcopy(NagradaKapaciteta)
NagradaSprememba_top = copy.deepcopy(NagradaSprememba)
NagradaPlacilo_top = copy.deepcopy(NagradaPlacilo)
train_agent_top = copy.deepcopy(train_agent)
train_env_top = copy.deepcopy(train_env)

for i in range(10):

    print(f'Ponovitev {i}')
    train = Learning_DQN(reset = True)
    nagradaDQN , placiloDQN, baterijaDQN, NagradaKapaciteta, NagradaSprememba, NagradaPlacilo = Learning_DQN(ucenje = False)
    nagradaDQN_stara = nagradaDQN[-1]
    print(f'Nagrada po optimizaciji {nagradaDQN[-1]}')
    print(f'Cena po optimizaciji {placiloDQN[-1]}')

    nagrada_seznam.append(nagradaDQN[-1])
    placilo_seznam.append(placiloDQN[-1])
    baterija_seznam.append(baterijaDQN[-1])
    NagradaKapaciteta_seznam.append(NagradaKapaciteta[-1])
    NagradaSprememba_seznam.append(NagradaSprememba[-1])
    NagradaPlacilo_seznam.append(NagradaPlacilo[-1])

    if (nagradaDQN[-1] > nagrada_top[-1]):
        nagrada_top = copy.deepcopy(nagradaDQN)
        placilo_top = copy.deepcopy(placiloDQN)
        baterija_top = copy.deepcopy(baterijaDQN)
        NagradaKapaciteta_top = copy.deepcopy(NagradaKapaciteta)
        NagradaSprememba_top = copy.deepcopy(NagradaSprememba)
        NagradaPlacilo_top = copy.deepcopy(NagradaPlacilo)
        train_agent_top = copy.deepcopy(train_agent)
        train_env_top = copy.deepcopy(train_env)

print(f'Najboljša nagrada: {nagrada_top[-1]}')
print(f'Cena pri najboljši nagradi: {placilo_top[-1]}')

In [ ]:
Y = [nagrada_seznam]
X = range(len(nagrada_seznam))
X_label = 'Ponovitev'
Y_label = 'Skupna Nagrada'
Legenda = ['Optimizirano DQN']
Naslov = 'Razlika med 10 ponovitvami DQN 1L 32N KP nagrada'
plotMultiY(X, Y, X_label, [Y_label], Legenda, Naslov, save_pdf = False)
# plotMultiY(X, Y, X_label, [Y_label], Legenda, Naslov, save_pdf = True)

In [ ]:
Y = [placilo_seznam]
X = range(len(placilo_seznam))
X_label = 'Ponovitev'
Y_label = 'Skupno placilo'
Legenda = ['Optimizirano DQN']
Naslov = 'Razlika med 10 ponovitvami DQN 1L 32N KP placilo'
plotMultiY(X, Y, X_label, [Y_label], Legenda, Naslov, save_pdf = False)
# plotMultiY(X, Y, X_label, [Y_label], Legenda, Naslov, save_pdf = True)